# 01 — From sky observables to Galactic phase space

**Scientific question:** We observe a star on the sky. How do those measurements become a statement such as: *this star is moving outward, rotating with the disc, and moving above the Galactic plane?*

> **Data boundary.** This notebook uses a tiny public **DESI DR1 MWS Iron Survey Validation 2 / bright** sample with Gaia DR3 crossmatches. It is not the DESI DR1 main-survey bright sample, not Lambert's DESI DR2 sample, and not representative population data for selection-function inference. Its sole purpose is a real-data coordinate lesson. See the [DESI DR1 MWS VAC overview](https://data.desi.lbl.gov/doc/releases/dr1/vac/mws/) and [official `rvpix` data model](https://desi-mws-dr1-datamodel.readthedocs.io/en/latest/rv_output/RVRUN/rvpix.html).

## 1. What DESI and Gaia actually measured

### Question
Which entries are measurements, and which quantities will we compute?

### Physical intuition
A position on the sky gives a direction, not a three-dimensional location. Gaia adds parallax and the two components of angular motion. DESI adds the motion along our line of sight. Together, these six phase-space coordinates allow a frame transformation.

- `ra`, `dec`: Gaia ICRS sky direction.
- `parallax`: the apparent annual displacement caused by Earth's orbit.
- `pmra`: $\mu_{\alpha *}=\dot{\alpha}\cos\delta$. The star's catalogue already includes the $\cos\delta$ factor; passing plain $\dot{\alpha}$ to Astropy would be wrong.
- `pmdec`: angular motion in declination.
- `radial_velocity`: DESI RVSpecFit line-of-sight velocity, positive when receding from the observer.

> **OBSERVABLE / MEASUREMENT:** the catalogue columns above and their uncertainties.  
> **COMPUTATIONAL OPERATION:** quality filtering, inverse parallax, and all coordinate transformations below.

In [ ]:
from pathlib import Path
import json

import astropy
import astropy.units as u
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import numpy as np
from astropy.coordinates import SkyCoord
from astropy.table import QTable

from lambert_lab.coordinates import (
    GALCEN_DISTANCE, GALCEN_V_SUN, Z_SUN, cylindrical_velocity_components,
    disc_azimuth, galactocentric_frame, transform_observables,
)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sample_path = ROOT / 'data' / 'derived' / 'public_demo_sample.ecsv'
provenance_path = ROOT / 'data' / 'derived' / 'public_demo_sample.provenance.json'
sample = QTable.read(sample_path)
provenance = json.loads(provenance_path.read_text())
print(f"Astropy {astropy.__version__}")
print(f"{len(sample)} teaching stars; source SHA-256 {sample.meta['source_sha256']}")
sample[['targetid', 'gaia_dr3_source_id', 'ra', 'dec', 'parallax', 'pmra', 'pmdec', 'radial_velocity', 'distance']][:5]

### Why this tiny sample?

The builder retains successful stellar DESI spectra with clean fibre status and $\sigma(v_{\rm los})\leq5$ km s$^{-1}$, then requires a five-parameter Gaia solution, at least eight visibility periods, `RUWE < 1.4`, no Gaia duplicated-source flag, $\varpi>0.5$ mas, and $\varpi/\sigma_\varpi\geq20$. `RUWE < 1.4` is a conventional heuristic, not a universal truth about Gaia solutions. Repeated Gaia source IDs are ranked by smallest radial-velocity uncertainty, highest R-arm S/N, then lowest DESI `TARGETID`; this file had no surviving repeats. A seeded random draw limits the lesson to 256 stars.

> **Distance caveat.** This sample is deliberately nearby and has positive, high-S/N parallaxes, so we use the pedagogical approximation $d[\mathrm{kpc}]=1/\varpi[\mathrm{mas}]$. This is not a complete treatment of Gaia systematic errors or distance posteriors. No Gaia parallax zero-point correction is applied. Never extrapolate this shortcut to the distant outer disc: Lambert's MSTO analysis is a fundamentally different distance regime.

In [ ]:
assert np.all(sample['parallax'] > 0.5 * u.mas)
assert np.all(sample['parallax_over_error'] >= 20)
assert u.allclose(sample['distance'], (1 / sample['parallax'].to_value(u.mas)) * u.kpc)
print('Cut flow (remaining rows):')
for item in provenance['cut_flow']:
    print(f"{item['remaining']:5d}  {item['cut']}")

## 2. Geometry: observer → Galactic → Galactocentric

### Physical intuition before equations
The observer-centred Galactic frame replaces $(\alpha,\delta)$ with longitude $l$ and latitude $b$. A Galactocentric transformation then changes the origin to the Galactic centre and accounts for the Sun's specified Galactocentric position and velocity. This is why an observed line-of-sight velocity is not, by itself, $V_R$.

The installed Astropy convention is checked directly here (see its [Galactocentric documentation](https://docs.astropy.org/en/stable/coordinates/galactocentric.html)). Astropy's Galactocentric Cartesian frame is right-handed: $x$ points approximately from the Sun toward the Galactic centre, so the Sun is at **negative** $x$; $y$ points roughly toward $l=90^\circ$; $z$ points to the North Galactic Pole. We define

$$R=\sqrt{x^2+y^2},\qquad \phi=\operatorname{atan2}(y,-x),$$

so $\phi=0$ on the Galactic-centre-to-Sun ray and positive $\phi$ follows disc rotation near the Sun. Then

$$V_R=\frac{xv_x+yv_y}{R},\qquad V_\phi=\frac{yv_x-xv_y}{R},\qquad V_Z=v_z.$$

Thus $V_R>0$ means outward, $V_\phi>0$ means prograde/disc rotation, and $V_Z>0$ means north/up. This disc-positive $V_\phi$ is the negative of the velocity along increasing `atan2(y, x)` in Astropy's axes. We use the Lambert-compatible disc-positive quantity $L_Z=R V_\phi$; it is positive for ordinary disc rotation. Lambert states a right-handed Galactocentric transformation and uses positive disc $V_\phi$ and $L_Z$; the paper does not explicitly define its plotted azimuth angle, so only these physical velocity meanings are claimed to be reconciled.

In [ ]:
frame = galactocentric_frame()
print(f'R0 = {GALCEN_DISTANCE}; z_sun = {Z_SUN}')
print(f'(v_x, v_y, v_z)_sun = {GALCEN_V_SUN}')
print(frame)

# Top-down coordinate/sign schematic (not a data figure).
fig, ax = plt.subplots(figsize=(7, 3.8))
sun_x = -GALCEN_DISTANCE.to_value(u.kpc)
ax.scatter([0, sun_x], [0, 0], s=[90, 70], c=['black', 'gold'], edgecolors='black', zorder=3)
ax.text(0, -0.35, 'Galactic centre', ha='center', va='top')
ax.text(sun_x, -0.35, 'Sun', ha='center', va='top')
ax.annotate('', xy=(2.2, 0), xytext=(0, 0), arrowprops=dict(arrowstyle='->', color='0.35', lw=1.7))
ax.text(2.25, 0.12, r'Astropy $+x$', color='0.35', ha='left')
ax.annotate('', xy=(0, 2.2), xytext=(0, 0), arrowprops=dict(arrowstyle='->', color='0.35', lw=1.7))
ax.text(0.15, 2.15, r'Astropy $+y$', color='0.35', va='top')
ax.annotate('', xy=(sun_x - 1.5, 0), xytext=(sun_x, 0), arrowprops=dict(arrowstyle='->', color='tab:blue', lw=2.2))
ax.text(sun_x - 0.75, -0.65, r'outward $+V_R$', color='tab:blue', ha='center')
ax.annotate('', xy=(sun_x, 1.7), xytext=(sun_x, 0), arrowprops=dict(arrowstyle='->', color='tab:red', lw=2.2))
ax.text(sun_x + 0.2, 1.55, r'prograde $+V_\phi$', color='tab:red', va='top')
ax.axhline(0, color='0.85', lw=0.8, zorder=0)
ax.set(xlim=(sun_x - 2.2, 3.0), ylim=(-1.2, 2.7),
       xlabel='Galactocentric x [kpc]', ylabel='Galactocentric y [kpc]',
       title='Top-down coordinate/sign schematic (not to scale)')
ax.set_aspect('equal')
fig.tight_layout()
plt.show()

## 3. One-star sign prediction — before transforming

### Prediction exercise
Imagine a synthetic star with the same Galactocentric azimuth as the Sun–anticentre line, farther from the Galactic centre than the Sun and 1 kpc above the plane: $(x,y,z)=(-10,0,1)$ kpc in Astropy's frame. Give it $(v_x,v_y,v_z)=(-30,220,15)$ km s$^{-1}$.

Before running the next cell, predict the signs. At negative $x$, a negative $v_x$ moves farther from the centre, so $V_R$ should be positive. Positive $v_y$ is disc rotation there, so $V_\phi$ should be positive. Positive $v_z$ points north, so $V_Z$ should be positive. We first turn this understandable Galactocentric construction into mock sky observables, then ask Astropy to recover it.

In [ ]:
synthetic_gc = SkyCoord(
    x=-10*u.kpc, y=0*u.kpc, z=1*u.kpc,
    v_x=-30*u.km/u.s, v_y=220*u.km/u.s, v_z=15*u.km/u.s,
    frame=frame, representation_type='cartesian', differential_type='cartesian',
)
mock_observed = synthetic_gc.transform_to('icrs')
print('Mock observables:', mock_observed)
recovered = mock_observed.transform_to(frame)
syn_vr, syn_vphi, syn_vz = cylindrical_velocity_components(
    recovered.x.to_value(u.kpc), recovered.y.to_value(u.kpc),
    recovered.v_x.to_value(u.km/u.s), recovered.v_y.to_value(u.km/u.s),
    recovered.v_z.to_value(u.km/u.s),
)
print(f'Recovered: V_R={syn_vr:.1f}, V_phi={syn_vphi:.1f}, V_Z={syn_vz:.1f} km/s')
assert np.allclose([syn_vr, syn_vphi, syn_vz], [30, 220, 15], atol=1e-9)

## 4. A real star, transparently

For one catalogue row we explicitly construct `SkyCoord`, rather than hiding the operation in a helper. Astropy interprets `pm_ra_cosdec` as $\mu_{\alpha *}$ and combines angular motion, distance, and line-of-sight velocity into a Cartesian velocity before changing the origin.

In [ ]:
star = sample[0]
one_sky = SkyCoord(
    ra=star['ra'], dec=star['dec'], distance=star['distance'],
    pm_ra_cosdec=star['pmra'], pm_dec=star['pmdec'],
    radial_velocity=star['radial_velocity'], frame='icrs',
)
one_galactic = one_sky.galactic
one_gc = one_sky.transform_to(frame)
one_R = np.hypot(one_gc.x, one_gc.y)
one_phi = disc_azimuth(one_gc.x, one_gc.y)
one_vr, one_vphi, one_vz = cylindrical_velocity_components(
    one_gc.x.to_value(u.kpc), one_gc.y.to_value(u.kpc),
    one_gc.v_x.to_value(u.km/u.s), one_gc.v_y.to_value(u.km/u.s),
    one_gc.v_z.to_value(u.km/u.s),
)
one_lz = one_R * one_vphi * u.km/u.s
print(f"l={one_galactic.l:.2f}, b={one_galactic.b:.2f}")
print(f"(x,y,z)=({one_gc.x:.3f}, {one_gc.y:.3f}, {one_gc.z:.3f})")
print(f"R={one_R:.3f}, phi={one_phi:.2f}")
print(f"(V_R,V_phi,V_Z)=({one_vr:.2f}, {one_vphi:.2f}, {one_vz:.2f}) km/s")
print(f"L_Z={one_lz.to(u.kpc*u.km/u.s):.1f}")
round_trip = one_gc.transform_to('icrs')
assert one_sky.separation_3d(round_trip).to_value(u.pc) < 1e-8

## 5. The same operation across the real sample

The helper below performs exactly the construction just shown and adds $l,b,x,y,z,R,\phi,V_R,V_\phi,V_Z,L_Z$ in memory. Those derived columns are intentionally absent from the committed ECSV so the notebook remains the place where the transformation happens.

In [ ]:
phase_space = transform_observables(sample)
assert phase_space['R'].unit == u.kpc
assert phase_space['V_phi'].unit == u.km/u.s
assert phase_space['L_Z'].unit == u.kpc*u.km/u.s
phase_space[['R', 'phi', 'z', 'V_R', 'V_phi', 'V_Z', 'L_Z']][:5]

In [ ]:
vr_values = phase_space['V_R'].to_value(u.km/u.s)
vmax = np.max(np.abs(vr_values))
vr_norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
fig, ax = plt.subplots(figsize=(7, 4.5))
points = ax.scatter(phase_space['R'].to_value(u.kpc), phase_space['V_phi'].to_value(u.km/u.s),
                    c=vr_values, cmap='coolwarm', norm=vr_norm, s=24, alpha=0.8)
ax.set(xlabel=r'$R$ [kpc]', ylabel=r'$V_\phi$ [km s$^{-1}$]',
       title=r'Nearby SV2 teaching stars ($V_R>0$: outward; $V_R<0$: inward)')
colorbar = fig.colorbar(points, ax=ax, label=r'$V_R$ [km s$^{-1}$]  ($>0$ outward; $<0$ inward)')
fig.tight_layout()
plt.show()

### What to notice

Most of this nearby, deliberately selected sample lies close to the solar radius; many stars have positive disc-rotation velocity, while $V_R$ spans both signs. This is a compact diagnostic of the transformed quantities, **not** a population measurement: SV2 targeting and our cuts are not a survey selection function.

> **DATA-SUPPORTED INFERENCE:** an individual star's signs describe its instantaneous motion under the adopted frame and distance estimate.  
> **MODEL-DEPENDENT INTERPRETATION:** explaining a multi-star ridge or wave requires dynamics and a controlled selection function; this sample does not establish such a structure.

## 6. 🧪 Try it yourself

Predict first: if the same star's measured line-of-sight velocity were 20 km s$^{-1}$ larger, would $V_R$, $V_\phi$, or both change? The answer depends on the sightline—the line-of-sight basis is generally not aligned with a Galactocentric cylindrical basis. Change `velocity_kick` and inspect the result.

In [ ]:
velocity_kick = 20 * u.km/u.s
experiment = sample[:1].copy()
baseline = transform_observables(experiment)
experiment['radial_velocity'] += velocity_kick
changed = transform_observables(experiment)
for component in ['V_R', 'V_phi', 'V_Z']:
    delta = (changed[component][0] - baseline[component][0]).to(u.km/u.s)
    print(f'Delta {component} = {delta:.2f}')

## 7. Bridge to Lambert's distant outer disc

We now know what the signs mean and how $L_Z=R V_\phi$ is constructed. Lambert et al. use these Galactocentric quantities to study distant MSTO stars and released downstream maps/waves. That science does **not** follow from this nearby SV2 sample, and the inverse-parallax shortcut used here must not be carried into Lambert's distance regime. Later notebooks use Lambert's released figure products, not this catalogue, and must distinguish measured patterns from dynamical interpretation—including any proposed link to Sagittarius.